In [1]:
from zipfile import ZipFile

import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

DATA_ZIP = "/Users/hanweyy/Downloads/RTH.zip"
RATE = 0.043
EXPIRY_HOUR = 16                      # options expire at the 16:00 close
SECONDS_PER_YEAR = 365 * 24 * 60 * 60
STRIKE_STEP = 25.0                    # spacing of the strike grid
QUOTE_FILES = ["put_bid", "put_ask", "call_bid", "call_ask"]

In [2]:
def read_quotes(day, expiry, name):
    with ZipFile(DATA_ZIP) as archive, archive.open(f"RTH/{day}_{name}_{expiry}.csv") as file:
        quotes = pd.read_csv(file, index_col=0)
    quotes.index = pd.to_datetime(quotes.index)
    quotes.columns = quotes.columns.astype(float)    # strikes as numbers, not text
    return quotes

def mid_price(bid, ask):
    return ((bid + ask) / 2).where((bid >= 0) & (ask >= 0))

def load_snapshot(day, expiry, time):
    quotes = {name: read_quotes(day, expiry, name) for name in QUOTE_FILES}
    row = quotes["put_bid"].index.get_indexer([pd.Timestamp(f"{day} {time}")], method="nearest")[0]
    snap = {name: frame.iloc[row] for name, frame in quotes.items()}
    options = pd.DataFrame({
        "Put_Mid": mid_price(snap["put_bid"], snap["put_ask"]),
        "Call_Mid": mid_price(snap["call_bid"], snap["call_ask"]),
    })
    return quotes["put_bid"].index[row], options


selected_time, options = load_snapshot("2025-04-01", "20250416", "15:55")
selected_time, options.loc[5600:5700]

(Timestamp('2025-04-01 15:55:01.126139'),
         Put_Mid  Call_Mid
 5600.0    76.10    120.50
 5625.0    85.25    104.70
 5650.0    95.35     89.80
 5675.0   106.40     75.95
 5700.0   119.25     62.50)

In [3]:
expiry_time = pd.to_datetime("20250416", format="%Y%m%d") + pd.Timedelta(hours=EXPIRY_HOUR)
T = (expiry_time - selected_time).total_seconds() / SECONDS_PER_YEAR
D = np.exp(-RATE * T)

both = options.dropna()
atm_strike = (both["Call_Mid"] - both["Put_Mid"]).abs().idxmin()
F = atm_strike + (both.loc[atm_strike, "Call_Mid"] - both.loc[atm_strike, "Put_Mid"]) / D
T, D, atm_strike, F

def put_curve(options, F, D):
    puts = options["Put_Mid"].copy()
    above = options.index > F
    puts[above] = options.loc[above, "Call_Mid"] - D * (F - options.index[above])
    return puts


puts = put_curve(options, F, D)
puts.loc[5600:5700].round(3)

cdf = (puts.shift(-1) - puts) / (STRIKE_STEP * D)
cdf.index = cdf.index + STRIKE_STEP / 2
cdf = cdf.dropna()

prob_fall_5 = np.interp(0.95 * F, cdf.index, cdf.values)
prob_fall_10 = np.interp(0.90 * F, cdf.index, cdf.values)
prob_fall_5, prob_fall_10

(np.float64(0.12205772681420365), np.float64(0.022839720393159114))

In [4]:
DENSITY_STEP = 2 * STRIKE_STEP    

density = (puts.shift(2) - 2 * puts + puts.shift(-2)) / (DENSITY_STEP**2 * D)
density = density.dropna()
density.loc[5550:5750].round(6)

5550.0    0.000922
5575.0    0.001483
5600.0    0.001603
5625.0    0.001505
5650.0    0.001347
5675.0    0.001525
5700.0    0.002224
5725.0    0.002264
5750.0    0.002044
Name: Put_Mid, dtype: float64

In [5]:
mass = (density * STRIKE_STEP).sum()
mean = (density.index * density * STRIKE_STEP).sum() / mass
negative_points = (density < 0).sum()

mass, mean, negative_points, len(density)

(np.float64(0.9949911545309407),
 np.float64(5652.164645928647),
 np.int64(4),
 68)

In [6]:
NO_QUOTE = -1.0
QUOTE_WINDOW = "5min"


def load_snapshot(day, expiry, time):
    quotes = {name: read_quotes(day, expiry, name) for name in QUOTE_FILES}
    centre = pd.Timestamp(f"{day} {time}")
    half = pd.Timedelta(QUOTE_WINDOW) / 2
    snap = {name: frame.loc[centre - half: centre + half].replace(NO_QUOTE, np.nan).median()
            for name, frame in quotes.items()}
    options = pd.DataFrame({
        "Put_Mid": (snap["put_bid"] + snap["put_ask"]) / 2,
        "Call_Mid": (snap["call_bid"] + snap["call_ask"]) / 2,
    })
    return centre, options

In [7]:
MOMENTS = pd.DataFrame([
    ("Apr 1 close",   "2025-04-01", "15:55", "20250416"),
    ("Apr 3 morning", "2025-04-03", "10:00", "20250416"),
    ("Apr 4 close",   "2025-04-04", "15:55", "20250416"),
    ("Apr 8 close",   "2025-04-08", "15:55", "20250515"),
    ("Apr 9 close",   "2025-04-09", "15:55", "20250515"),
], columns=["Moment", "Day", "Time", "Expiry"])


def forward_from_parity(options, D):
    both = options.dropna()
    atm_strike = (both["Call_Mid"] - both["Put_Mid"]).abs().idxmin()
    return atm_strike + (both.loc[atm_strike, "Call_Mid"] - both.loc[atm_strike, "Put_Mid"]) / D


def analyze_moment(moment):
    centre, options = load_snapshot(moment["Day"], moment["Expiry"], moment["Time"])
    expiry_time = pd.to_datetime(moment["Expiry"], format="%Y%m%d") + pd.Timedelta(hours=EXPIRY_HOUR)
    T = (expiry_time - centre).total_seconds() / SECONDS_PER_YEAR
    D = np.exp(-RATE * T)
    F = forward_from_parity(options, D)
    puts = put_curve(options, F, D)

    cdf = (puts.shift(-1) - puts) / (STRIKE_STEP * D)
    cdf.index = cdf.index + STRIKE_STEP / 2
    cdf = cdf.dropna()
    density = ((puts.shift(2) - 2 * puts + puts.shift(-2)) / (DENSITY_STEP**2 * D)).dropna()

    mass = (density * STRIKE_STEP).sum()
    # A 10% fall is only readable if 90% of F lies inside the strike grid.
    readable_10 = 0.90 * F >= cdf.index[0]
    summary = {
        "Moment": moment["Moment"], "Days_to_expiry": T * 365, "Forward": F,
        "P_fall_5pct": np.interp(0.95 * F, cdf.index, cdf.values),
        "P_fall_10pct": np.interp(0.90 * F, cdf.index, cdf.values) if readable_10 else np.nan,
        "P_below_grid": cdf.iloc[0],
        "Density_mass": mass,
        "Density_mean": (density.index * density * STRIKE_STEP).sum() / mass,
        "Negative_points": (density < 0).sum(),
    }
    return summary, cdf, density


results = [analyze_moment(moment) for _, moment in MOMENTS.iterrows()]
part_c = pd.DataFrame([summary for summary, _, _ in results])
part_c.round(3)

,Moment,Days_to_expiry,Forward,P_fall_5pct,P_fall_10pct,P_below_grid,Density_mass,Density_mean,Negative_points
0,Apr 1 close,15.003,5639.381,0.121,0.022,0.004,0.994,5647.015,2
1,Apr 3 morning,13.250,5476.127,0.145,0.032,0.008,0.991,5490.631,1
2,Apr 4 close,12.003,5088.834,0.217,NaN,0.136,0.863,5224.632,5
3,Apr 8 close,37.003,4985.797,0.269,NaN,0.227,0.763,5291.884,0
4,Apr 9 close,36.003,5483.637,0.205,0.103,0.040,0.946,5561.679,2


In [8]:
moment_order = list(MOMENTS["Moment"])
forwards = part_c.set_index("Moment")["Forward"]


def to_long(results, curve_index, value_name):
    frames = []
    for summary, *curves in results:
        curve = curves[curve_index]
        frames.append(pd.DataFrame({
            "Moment": summary["Moment"],
            "Moneyness": curve.index / summary["Forward"] * 100,
            value_name: curve.values,
        }))
    return pd.concat(frames, ignore_index=True)


cdf_long = to_long(results, 0, "CDF")
density_long = to_long(results, 1, "Density")
density_long["Density"] = density_long["Density"] * density_long["Moment"].map(forwards) / 100

In [9]:
#CDF Chart
(ggplot(cdf_long, aes("Moneyness", "CDF", color="Moment"))
 + geom_line(size=1)
 + geom_vline(xintercept=95, linetype="dashed", color="gray")
 + scale_color_discrete(limits=moment_order)
 + labs(title="Market probability that SPX ends below each level",
        subtitle="Risk-neutral CDF; dashed line = a 5% fall from the forward",
        x="Strike / forward (%)", y="Probability SPX ends below", color="")
 + theme_minimal() + ggsize(850, 450))

In [10]:
#Density chart
(ggplot(density_long, aes("Moneyness", "Density", color="Moment"))
 + geom_line(size=1)
 + geom_hline(yintercept=0, color="gray")
 + scale_color_discrete(limits=moment_order)
 + labs(title="Risk-neutral density of SPX at expiry (Breeden-Litzenberger)",
        subtitle="Probability per 1% of the forward; curves stop at the lowest strike (4600)",
        x="Strike / forward (%)", y="Probability per 1% move", color="")
 + theme_minimal() + ggsize(850, 450))


In [12]:
probs = part_c.melt(id_vars="Moment", value_vars=["P_fall_5pct", "P_fall_10pct"],
                    var_name="Fall", value_name="Probability")
probs["Fall"] = probs["Fall"].map({"P_fall_5pct": "Fall of 5% or more",
                                   "P_fall_10pct": "Fall of 10% or more"})

(ggplot(probs.dropna(), aes("Moment", "Probability", fill="Fall"))
 + geom_bar(stat="identity", position="dodge")
 + scale_x_discrete(limits=moment_order)
 + labs(title="Market-implied crash probabilities",
        subtitle="10% fall not readable on Apr 4 and Apr 8: below the lowest strike",
        x="", y="Risk-neutral probability", fill="")
 + theme_minimal() + ggsize(850, 450))